# NEXUS-KO · interface **energy** vs co-dependency (AlphaFold-Multimer + PRODIGY, GPU)

Of 55 top obligate pairs, **38 already have a fetched structure** (34 solved PDB + 4 AFDB pre-computed AF-Multimer heterodimers) — no GPU for those. **This notebook predicts only the 17 pairs with no public structure** (AF-Multimer's low-confidence tail: integrins, mTORC2, HOPS/COG/MICOS). For each: **predict the complex (ColabFold)**, compute a real interface **binding energy ΔG (PRODIGY)**, and test whether interface *energy* predicts DepMap co-dependency — the question crude interface size failed (Spearman 0.17, p=0.32 on the 38 fetched).

**Autosaves to Google Drive** at `MyDrive/nexus_ko/`; disconnect-safe and **resumes**.

**Setup:** `Runtime → Change runtime type → GPU`. ~17 pairs: L4 ~1 h; 5-pair test ~30 min.

**Bounds:** protein-level readout vs co-dependency — not the mRNA far field, not the wall. ipTM/ΔG are estimates.

In [ ]:
!nvidia-smi -L


## 1 · Mount Google Drive (autosave target)

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/nexus_ko'
else:
    WORK = '/content/nexus_ko'
AF_IN, AF_OUT, RESULT_JSON = f'{WORK}/af_in', f'{WORK}/af_out', f'{WORK}/nexus_ko_afmultimer.json'
for d in (WORK, AF_IN, AF_OUT): os.makedirs(d, exist_ok=True)
print('autosaving to', WORK)


## 2 · Install ColabFold + PRODIGY

In [ ]:
import os
if not os.path.exists(f'{WORK}/.installed'):
    os.system('pip -q install "colabfold[alphafold]" 2>/dev/null')
    os.system('pip -q install prodigy-prot biopython scipy 2>/dev/null')
    open(f'{WORK}/.installed','w').close()
print('colabfold_batch:', os.popen('which colabfold_batch').read().strip() or 'NOT FOUND')


## 3 · Pairs (embedded Python literal). `pdb`=already fetched (skipped).

In [ ]:
PAIRS = [{'a': 'TSC1', 'b': 'TSC2', 'codep': 0.912, 'pdb': '7DL2', 'real_iface': 200, 'source': 'PDB'}, {'a': 'DEPDC5', 'b': 'NPRL2', 'codep': 0.799, 'pdb': '6CES', 'real_iface': 77, 'source': 'PDB'}, {'a': 'SDHA', 'b': 'SDHB', 'codep': 0.794, 'pdb': '8GS8', 'real_iface': 127, 'source': 'PDB'}, {'a': 'RNASEH2A', 'b': 'RNASEH2C', 'codep': 0.789, 'pdb': '3P56', 'real_iface': 120, 'source': 'PDB'}, {'a': 'AP2M1', 'b': 'AP2S1', 'codep': 0.781, 'pdb': '6URI', 'real_iface': 9, 'source': 'PDB'}, {'a': 'POLE3', 'b': 'POLE4', 'codep': 0.776, 'pdb': 'AF-0000000203705504', 'real_iface': 109, 'source': 'AF-Multimer(AFDB)'}, {'a': 'WDR26', 'b': 'YPEL5', 'codep': 0.762, 'pdb': '8QBN', 'real_iface': 58, 'source': 'PDB'}, {'a': 'PDHA1', 'b': 'PDHB', 'codep': 0.758, 'pdb': '1NI4', 'real_iface': 512, 'source': 'PDB'}, {'a': 'HSD17B10', 'b': 'PRORP', 'codep': 0.757, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'POLG', 'b': 'POLG2', 'codep': 0.745, 'pdb': '3IKM', 'real_iface': 234, 'source': 'PDB'}, {'a': 'MAPKAP1', 'b': 'RICTOR', 'codep': 0.744, 'pdb': '5ZCS', 'real_iface': 42, 'source': 'PDB'}, {'a': 'GATB', 'b': 'QRSL1', 'codep': 0.736, 'pdb': 'AF-0000000210489125', 'real_iface': 89, 'source': 'AF-Multimer(AFDB)'}, {'a': 'RGP1', 'b': 'RIC1', 'codep': 0.736, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'MIOS', 'b': 'WDR24', 'codep': 0.733, 'pdb': '7UHY', 'real_iface': 145, 'source': 'PDB'}, {'a': 'KCMF1', 'b': 'UBR4', 'codep': 0.733, 'pdb': '9NWE', 'real_iface': 202, 'source': 'PDB'}, {'a': 'NCAPG2', 'b': 'NCAPH2', 'codep': 0.729, 'pdb': '9F5W', 'real_iface': 85, 'source': 'PDB'}, {'a': 'EED', 'b': 'EZH2', 'codep': 0.728, 'pdb': '5GSA', 'real_iface': 144, 'source': 'PDB'}, {'a': 'BRD9', 'b': 'SMARCD1', 'codep': 0.727, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'EXT1', 'b': 'EXT2', 'codep': 0.727, 'pdb': '7SCH', 'real_iface': 181, 'source': 'PDB'}, {'a': 'PARD3', 'b': 'PARD6B', 'codep': 0.726, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'GET1', 'b': 'GET3', 'codep': 0.726, 'pdb': '6SO5', 'real_iface': 63, 'source': 'PDB'}, {'a': 'PSMG1', 'b': 'PSMG2', 'codep': 0.725, 'pdb': '8QYJ', 'real_iface': 72, 'source': 'PDB'}, {'a': 'NCAPD3', 'b': 'NCAPH2', 'codep': 0.723, 'pdb': '9F5W', 'real_iface': 119, 'source': 'PDB'}, {'a': 'DLST', 'b': 'OGDH', 'codep': 0.706, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'TGFBR1', 'b': 'TGFBR2', 'codep': 0.704, 'pdb': '2PJY', 'real_iface': 25, 'source': 'PDB'}, {'a': 'GATB', 'b': 'GATC', 'codep': 0.702, 'pdb': 'AF-0000000203794375', 'real_iface': 102, 'source': 'AF-Multimer(AFDB)'}, {'a': 'MAU2', 'b': 'NIPBL', 'codep': 0.701, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'STN1', 'b': 'TEN1', 'codep': 0.701, 'pdb': '4JOI', 'real_iface': 191, 'source': 'PDB'}, {'a': 'RAD51D', 'b': 'XRCC2', 'codep': 0.697, 'pdb': '8FAZ', 'real_iface': 108, 'source': 'PDB'}, {'a': 'MIOS', 'b': 'WDR59', 'codep': 0.696, 'pdb': '7UHY', 'real_iface': 130, 'source': 'PDB'}, {'a': 'NCAPD3', 'b': 'NCAPG2', 'codep': 0.694, 'pdb': '9F5W', 'real_iface': 21, 'source': 'PDB'}, {'a': 'VPS18', 'b': 'VPS33A', 'codep': 0.694, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'FANCD2', 'b': 'FANCI', 'codep': 0.694, 'pdb': '6VAA', 'real_iface': 111, 'source': 'PDB'}, {'a': 'METTL1', 'b': 'WDR4', 'codep': 0.685, 'pdb': '7U20', 'real_iface': 49, 'source': 'PDB'}, {'a': 'EED', 'b': 'SUZ12', 'codep': 0.685, 'pdb': '4W2R', 'real_iface': 62, 'source': 'PDB'}, {'a': 'PNPT1', 'b': 'SUPV3L1', 'codep': 0.685, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'HUS1', 'b': 'RAD9A', 'codep': 0.683, 'pdb': '3A1J', 'real_iface': 55, 'source': 'PDB'}, {'a': 'EZH2', 'b': 'SUZ12', 'codep': 0.683, 'pdb': '5HYN', 'real_iface': 551, 'source': 'PDB'}, {'a': 'ITGAV', 'b': 'ITGB5', 'codep': 0.678, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'RNASEH2A', 'b': 'RNASEH2B', 'codep': 0.675, 'pdb': '3P56', 'real_iface': 77, 'source': 'PDB'}, {'a': 'MLST8', 'b': 'RICTOR', 'codep': 0.675, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'SDHB', 'b': 'SDHC', 'codep': 0.674, 'pdb': '8GS8', 'real_iface': 81, 'source': 'PDB'}, {'a': 'PARD6B', 'b': 'PRKCI', 'codep': 0.669, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'MICOS10', 'b': 'MICOS13', 'codep': 0.666, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'PRORP', 'b': 'TRMT10C', 'codep': 0.659, 'pdb': '7ONU', 'real_iface': 45, 'source': 'PDB'}, {'a': 'MAEA', 'b': 'WDR26', 'codep': 0.657, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'CBFB', 'b': 'RUNX1', 'codep': 0.656, 'pdb': '1E50', 'real_iface': 332, 'source': 'PDB'}, {'a': 'COG5', 'b': 'COG7', 'codep': 0.655, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'ACTR2', 'b': 'ARPC4', 'codep': 0.654, 'pdb': '6UHC', 'real_iface': 37, 'source': 'PDB'}, {'a': 'TUBD1', 'b': 'TUBE1', 'codep': 0.653, 'pdb': 'AF-0000000211872185', 'real_iface': 117, 'source': 'AF-Multimer(AFDB)'}, {'a': 'CABIN1', 'b': 'HIRA', 'codep': 0.651, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'SDHA', 'b': 'SDHC', 'codep': 0.647, 'pdb': '8GS8', 'real_iface': 3, 'source': 'PDB'}, {'a': 'VPS39', 'b': 'VPS41', 'codep': 0.647, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'PEX26', 'b': 'PEX6', 'codep': 0.645, 'pdb': None, 'real_iface': None, 'source': None}, {'a': 'RNASEH2B', 'b': 'RNASEH2C', 'codep': 0.642, 'pdb': '3P56', 'real_iface': 216, 'source': 'PDB'}]
TO_PREDICT = [p for p in PAIRS if not p['pdb']]          # only pairs with no fetched structure
N_PAIRS = min(20, len(TO_PREDICT))
work = sorted(TO_PREDICT, key=lambda p: -p['codep'])[:N_PAIRS]
print(f'{sum(1 for p in PAIRS if p["pdb"])} already fetched; predicting {len(work)} of {len(TO_PREDICT)} unfetchable pairs')


## 4 · Sequences → multimer FASTAs (Drive; skips already-predicted)

In [ ]:
import requests, glob
def gene_seq(g):
    r=requests.get('https://rest.uniprot.org/uniprotkb/search',params={'query':f'gene_exact:{g} AND organism_id:9606 AND reviewed:true','fields':'sequence','format':'fasta'},timeout=30)
    return ''.join(r.text.split('\n')[1:]).strip() if (r.status_code==200 and r.text.startswith('>')) else None
seqs={}
for p in work:
    for g in (p['a'],p['b']):
        if g not in seqs: seqs[g]=gene_seq(g)
written=[]
for p in work:
    name=f"{p['a']}__{p['b']}"
    if glob.glob(f'{AF_OUT}/{name}_*rank_001*.pdb'): written.append((name,p)); continue
    sa,sb=seqs.get(p['a']),seqs.get(p['b'])
    if not sa or not sb or len(sa)+len(sb)>1800: continue
    open(f'{AF_IN}/{name}.fasta','w').write(f'>{name}\n{sa}:{sb}\n'); written.append((name,p))
print('ready:',len(written),'pairs')


## 5 · Run AlphaFold-Multimer → autosave to Drive (resume-safe)

In [ ]:
os.system(f'colabfold_batch "{AF_IN}" "{AF_OUT}" --num-models 1 --num-recycle 3 --rank iptm 2>&1 | tail -5')
print('saved to',AF_OUT)


## 6 · Interface energy (PRODIGY ΔG) + ipTM — autosaved incrementally

In [ ]:
import json, glob, re, numpy as np
def top_pdb(n): h=sorted(glob.glob(f'{AF_OUT}/{n}_*rank_001*.pdb')+glob.glob(f'{AF_OUT}/{n}*rank_1*.pdb')); return h[0] if h else None
def sc(n):
    j=sorted(glob.glob(f'{AF_OUT}/{n}_*rank_001*.json')+glob.glob(f'{AF_OUT}/{n}*scores*rank_1*.json'))
    if not j: return {}
    d=json.load(open(j[0])); return {'iptm':d.get('iptm'),'pae':d.get('pae') or d.get('predicted_aligned_error')}
def dg(pdb):
    o=os.popen(f'prodigy "{pdb}" --selection A B -q 2>/dev/null').read().strip()
    m=re.search(r'(-?\d+\.\d+)', o.split(chr(10))[-1]) if o else None
    return float(m.group(1)) if m else None
rows=[]
for name,p in written:
    pdb=top_pdb(name)
    if not pdb: continue
    s=sc(name); g=dg(pdb); ipae=None
    try:
        la=len(seqs[p['a']]); pae=np.array(s['pae']); ipae=float((pae[:la,la:].mean()+pae[la:,:la].mean())/2)
    except Exception: pass
    rows.append({**p,'iptm':s.get('iptm'),'interface_pae':ipae,'prodigy_dG':g,'pdb_file':os.path.basename(pdb)})
    json.dump(rows,open(RESULT_JSON,'w'),indent=1)
    print(f"{p['a']:9s}-{p['b']:9s} codep={p['codep']:.2f} ipTM={s.get('iptm')} dG={g}")
print('saved',len(rows),'->',RESULT_JSON)


## 7 · Does interface **energy** predict co-dependency?

In [ ]:
import numpy as np
from scipy.stats import spearmanr
for metric,sign in [('prodigy_dG',-1),('iptm',1),('interface_pae',-1)]:
    v=[(r[metric],r['codep']) for r in rows if r.get(metric) is not None]
    if len(v)>=6:
        a,c=zip(*v); rho,pp=spearmanr([sign*x for x in a],c); print(f'{metric:14s} vs co-dependency: rho={rho:+.3f} (p={pp:.3f}, n={len(v)})')
print('\nCompare to the fetched-structure size result (rho~0.17, p=0.32): if energy beats it, interface STRENGTH sharpens obligate prediction where size did not.')


## 8 · Results in `MyDrive/nexus_ko/`
Hand `nexus_ko_afmultimer.json` back to the sandbox to merge with the 38 fetched.